In [ ]:
"""
step3_q2_content_nlp.py
======================
Step 3 — Question 2 content exploration

This script reads the Step 1 interview outputs and prepares the text-content
materials for Question 2. It does not depend on Step 2 metadata extraction.

Primary input priority:
1. output/interview_data.xlsx
2. output/raw_transcripts.csv

Outputs:
  - q2_filtered_dialogue.csv
  - q2_term_frequency.csv
  - q2_tfidf_top_terms.csv
  - q2_top_terms.png
  - q2_wordcloud.png

Usage:
    Local computer:
        .venv/bin/python3 scripts/step3_q2_content_nlp.py

    Google Colab:
        from google.colab import drive
        drive.mount('/content/drive')
        %cd "/content/drive/MyDrive/NLP for Oral history"
        !python scripts/step3_q2_content_nlp.py --project-root "/content/drive/MyDrive/NLP for Oral history"
"""

from __future__ import annotations

import argparse
import os
import re
from collections import Counter
from pathlib import Path

import pandas as pd

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _MPL_AVAILABLE = True
except ImportError:
    _MPL_AVAILABLE = False

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    _SKLEARN_AVAILABLE = True
except ImportError:
    _SKLEARN_AVAILABLE = False

try:
    from wordcloud import WordCloud
    _WORDCLOUD_AVAILABLE = True
except ImportError:
    _WORDCLOUD_AVAILABLE = False


PROJECT_ROOT = Path(os.environ.get("OH_PROJECT_ROOT", Path(__file__).resolve().parents[1]))
OUTPUT_DIR = Path(os.environ.get("OH_OUTPUT_DIR", PROJECT_ROOT / "output"))
DEFAULT_COLAB_ROOT = Path("/content/drive/MyDrive/NLP for Oral history")
INPUT_XLSX = OUTPUT_DIR / "interview_data.xlsx"
INPUT_CSV = OUTPUT_DIR / "raw_transcripts.csv"

OUT_FILTERED = OUTPUT_DIR / "q2_filtered_dialogue.csv"
OUT_TERM_FREQ = OUTPUT_DIR / "q2_term_frequency.csv"
OUT_TFIDF = OUTPUT_DIR / "q2_tfidf_top_terms.csv"
OUT_TOP_TERMS_PLOT = OUTPUT_DIR / "q2_top_terms.png"
OUT_WORDCLOUD = OUTPUT_DIR / "q2_wordcloud.png"


def resolve_runtime_project_root(mode: str, explicit_project_root: str | None) -> str | None:
    """
    Resolve a project root that works both locally and in Google Colab.
    Priority:
      1. --project-root
      2. OH_PROJECT_ROOT
      3. default Google Drive path when --mode colab is used
      4. None -> keep local repo-relative default
    """
    if explicit_project_root:
        return explicit_project_root
    env_root = os.environ.get("OH_PROJECT_ROOT")
    if env_root:
        return env_root
    if mode in {"colab", "original_colab"} and DEFAULT_COLAB_ROOT.exists():
        return str(DEFAULT_COLAB_ROOT)
    return None


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Step 3 — Question 2 content NLP.")
    parser.add_argument(
        "--mode",
        choices=["local", "colab", "original_colab"],
        default="local",
        help="Runtime mode. Colab can auto-resolve the default Google Drive project path.",
    )
    parser.add_argument("--project-root", help="Project root containing output/.")
    parser.add_argument("--output-dir", help="Output directory.")
    parser.add_argument("--input-file", help="Explicit Step 1 interview-data file.")
    return parser.parse_args()


def configure_paths(
    project_root: str | None = None,
    output_dir: str | None = None,
    input_file: str | None = None,
) -> Path:
    global PROJECT_ROOT, OUTPUT_DIR, INPUT_XLSX, INPUT_CSV
    global OUT_FILTERED, OUT_TERM_FREQ, OUT_TFIDF, OUT_TOP_TERMS_PLOT, OUT_WORDCLOUD

    if project_root:
        PROJECT_ROOT = Path(project_root).expanduser().resolve()
    if output_dir:
        OUTPUT_DIR = Path(output_dir).expanduser().resolve()
    elif project_root:
        OUTPUT_DIR = PROJECT_ROOT / "output"

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    INPUT_XLSX = OUTPUT_DIR / "interview_data.xlsx"
    INPUT_CSV = OUTPUT_DIR / "raw_transcripts.csv"
    OUT_FILTERED = OUTPUT_DIR / "q2_filtered_dialogue.csv"
    OUT_TERM_FREQ = OUTPUT_DIR / "q2_term_frequency.csv"
    OUT_TFIDF = OUTPUT_DIR / "q2_tfidf_top_terms.csv"
    OUT_TOP_TERMS_PLOT = OUTPUT_DIR / "q2_top_terms.png"
    OUT_WORDCLOUD = OUTPUT_DIR / "q2_wordcloud.png"

    if input_file:
        return Path(input_file).expanduser().resolve()
    if INPUT_XLSX.exists():
        return INPUT_XLSX
    if INPUT_CSV.exists():
        return INPUT_CSV
    raise FileNotFoundError(
        "No Step 1 interview-data file found. Expected output/interview_data.xlsx "
        "or output/raw_transcripts.csv. Run step1_read.py first."
    )


def load_step1_result(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    return pd.read_csv(path)


def _first_present(df: pd.DataFrame, *names: str) -> str | None:
    for name in names:
        if name in df.columns:
            return name
    return None


def _clean_text(text: object) -> str:
    if pd.isna(text):
        return ""
    s = str(text)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


STOPWORDS = {
    "the", "and", "that", "have", "with", "this", "from", "they", "were", "when",
    "what", "your", "about", "would", "there", "their", "them", "then", "just",
    "like", "because", "really", "very", "into", "here", "been", "also", "where",
    "which", "while", "after", "before", "could", "should", "going", "through",
    "them", "than", "some", "more", "much", "many", "only", "over", "well",
    "said", "says", "told", "tell", "know", "think", "yeah", "okay", "uh", "uhh",
    "haha", "went", "come", "came", "back", "still", "want", "wanted", "made",
    "make", "made", "doing", "done", "used", "using", "would", "didn", "dont",
    "cant", "im", "ive", "hes", "shes", "wasnt", "werent", "youre", "thats",
    "ietnamese", "american", "project", "interview", "narrator", "interviewer",
}


def _tokenize(text: str) -> list[str]:
    tokens = re.findall(r"[a-zA-ZÀ-ỹ]{3,}", text.lower())
    return [t for t in tokens if t not in STOPWORDS]


def build_filtered_dialogue(df: pd.DataFrame) -> pd.DataFrame:
    id_col = _first_present(df, "Interview_ID", "interview_id")
    name_col = _first_present(df, "Narrator_Name", "narrator_name")
    narrator_col = _first_present(df, "narrator_dialogue", "Narrator_Dialogue")
    interviewer_col = _first_present(df, "interviewer_dialogue", "Interviewer_Dialogue")

    if not id_col or not narrator_col:
        raise ValueError("Step 1 result is missing Interview_ID/interview_id or narrator_dialogue.")

    out = pd.DataFrame({
        "Interview_ID": df[id_col].astype(str),
        "Narrator_Name": df[name_col].astype(str) if name_col else "",
        "narrator_dialogue": df[narrator_col].map(_clean_text),
        "interviewer_dialogue": df[interviewer_col].map(_clean_text) if interviewer_col else "",
    })
    out = out[out["narrator_dialogue"].str.strip().ne("")].copy()
    out["token_count"] = out["narrator_dialogue"].map(lambda x: len(_tokenize(x)))
    return out


def corpus_term_frequency(dialogue_df: pd.DataFrame) -> pd.DataFrame:
    counter: Counter[str] = Counter()
    for text in dialogue_df["narrator_dialogue"]:
        counter.update(_tokenize(text))
    term_df = pd.DataFrame(counter.items(), columns=["term", "frequency"])
    term_df = term_df.sort_values("frequency", ascending=False).reset_index(drop=True)
    return term_df


def per_interview_tfidf(dialogue_df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    if not _SKLEARN_AVAILABLE or dialogue_df.empty:
        return pd.DataFrame(columns=["Interview_ID", "Narrator_Name", "term", "tfidf"])

    vec = TfidfVectorizer(
        lowercase=True,
        stop_words=list(STOPWORDS),
        token_pattern=r"(?u)\b[a-zA-ZÀ-ỹ]{3,}\b",
        max_features=5000,
    )
    mat = vec.fit_transform(dialogue_df["narrator_dialogue"])
    vocab = vec.get_feature_names_out()

    rows = []
    for i, row in dialogue_df.reset_index(drop=True).iterrows():
        dense = mat[i].toarray().ravel()
        if dense.size == 0 or dense.max() <= 0:
            continue
        top_idx = dense.argsort()[::-1][:top_n]
        for idx in top_idx:
            score = float(dense[idx])
            if score <= 0:
                continue
            rows.append({
                "Interview_ID": row["Interview_ID"],
                "Narrator_Name": row["Narrator_Name"],
                "term": vocab[idx],
                "tfidf": round(score, 6),
            })
    return pd.DataFrame(rows)


def plot_top_terms(term_df: pd.DataFrame, top_n: int = 20) -> None:
    if not _MPL_AVAILABLE or term_df.empty:
        return
    top = term_df.head(top_n).iloc[::-1]
    _, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top["term"], top["frequency"], color="#2D6A8E")
    ax.set_xlabel("Frequency")
    ax.set_ylabel("Term")
    ax.set_title("Question 2: Top Narrator Terms", fontweight="bold")
    ax.grid(axis="x", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_TOP_TERMS_PLOT, dpi=150)
    plt.close()


def plot_wordcloud(term_df: pd.DataFrame) -> None:
    if not (_MPL_AVAILABLE and _WORDCLOUD_AVAILABLE) or term_df.empty:
        return
    freq = dict(zip(term_df["term"], term_df["frequency"]))
    wc = WordCloud(width=1600, height=900, background_color="white", colormap="viridis")
    wc.generate_from_frequencies(freq)
    _, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title("Question 2: Narrator Word Cloud", fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUT_WORDCLOUD, dpi=150)
    plt.close()


if __name__ == "__main__":
    args = parse_args()
    resolved_project_root = resolve_runtime_project_root(args.mode, args.project_root)
    input_path = configure_paths(
        project_root=resolved_project_root,
        output_dir=args.output_dir,
        input_file=args.input_file,
    )

    print(f"Mode: {args.mode}")
    print(f"Project root: {PROJECT_ROOT}")
    print(f"Output dir: {OUTPUT_DIR}")
    print(f"Step 1 input: {input_path}")

    raw = load_step1_result(input_path)
    dialogue_df = build_filtered_dialogue(raw)
    dialogue_df.to_csv(OUT_FILTERED, index=False, encoding="utf-8-sig")
    print(f"Saved filtered dialogue: {OUT_FILTERED}")
    print(f"Usable narrator rows: {len(dialogue_df)}")

    term_df = corpus_term_frequency(dialogue_df)
    term_df.to_csv(OUT_TERM_FREQ, index=False, encoding="utf-8-sig")
    print(f"Saved corpus term frequency: {OUT_TERM_FREQ}")

    tfidf_df = per_interview_tfidf(dialogue_df)
    tfidf_df.to_csv(OUT_TFIDF, index=False, encoding="utf-8-sig")
    print(f"Saved TF-IDF terms: {OUT_TFIDF}")

    plot_top_terms(term_df)
    if _MPL_AVAILABLE and term_df.shape[0] > 0:
        print(f"Saved top-terms plot: {OUT_TOP_TERMS_PLOT}")
    elif not _MPL_AVAILABLE:
        print("[WARN] matplotlib not installed; skipped top-terms plot.")

    plot_wordcloud(term_df)
    if _WORDCLOUD_AVAILABLE and _MPL_AVAILABLE and term_df.shape[0] > 0:
        print(f"Saved word cloud: {OUT_WORDCLOUD}")
    elif not _WORDCLOUD_AVAILABLE:
        print("[WARN] wordcloud not installed; skipped word cloud.")
